# Actividad 6: Detección de Tumores Cerebrales con Redes Neuronales Profundas

**Objetivo:** Desarrollar un modelo de Deep Learning basado en VGG16 (Transfer Learning) para detectar tumores cerebrales a partir de imágenes de MRI.

**Dataset:** `brain-mri-images-for-brain-tumor-detection` (Kaggle)  
**Clases:** `YES` (tumor presente) | `NO` (sin tumor)

## 1. Instalación e importación de librerías

Se instala `imutils` (no incluida por defecto en Kaggle) y se importan todas las dependencias necesarias para procesamiento de imágenes, visualización y construcción del modelo.

In [10]:
!pip install imutils -q

import numpy as np
from tqdm import tqdm
import cv2
import os
import shutil
import itertools
import imutils
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras import layers
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_SEED = 123
print("Librerías importadas correctamente")

Librerías importadas correctamente


## 2. Preparación del entorno de directorios

Se crean las carpetas `TRAIN`, `TEST` y `VAL`, cada una con subcarpetas `YES` y `NO`, donde se almacenarán las imágenes separadas por conjunto.

In [11]:
import os

# Cambiar al directorio del notebook para que las rutas relativas funcionen
try:
    os.chdir(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)))
except NameError:
    pass  # Si no está disponible, usar el directorio actual

# Crear estructura de directorios
for split in ['TRAIN', 'TEST', 'VAL']:
    for cls in ['YES', 'NO']:
        os.makedirs(f'{split}/{cls}', exist_ok=True)

print(f"Directorio de trabajo: {os.getcwd()}")
print("Estructura de directorios creada:")
for split in ['TRAIN', 'TEST', 'VAL']:
    print(f"  {split}/")
    for cls in ['YES', 'NO']:
        print(f"    {cls}/")

Directorio de trabajo: /content
Estructura de directorios creada:
  TRAIN/
    YES/
    NO/
  TEST/
    YES/
    NO/
  VAL/
    YES/
    NO/


## 3. División del dataset en Train / Test / Val

El dataset original contiene ~98 imágenes clase NO y ~155 imágenes clase YES.  
Se usa el **80%** para entrenamiento y el resto se divide entre prueba y validación.

In [20]:
IMG_PATH = 'archive/'

for CLASS in os.listdir(IMG_PATH):
    if not os.path.isfile(CLASS):
        if not os.path.isfile(CLASS) and (CLASS == 'yes' or CLASS == 'no'):
            print(CLASS)
            IMG_NUM = len(os.listdir(IMG_PATH + CLASS))
            print(f'Total imágenes en clase {CLASS}: {IMG_NUM}')
            for n, FILE_NAME in enumerate(os.listdir(IMG_PATH + CLASS)):
                img = IMG_PATH + CLASS + '/' + FILE_NAME
                if n < 5:
                    shutil.copy(img, 'TEST/' + CLASS.upper() + '/' + FILE_NAME)
                elif n < 0.8 * IMG_NUM:
                    shutil.copy(img, 'TRAIN/' + CLASS.upper() + '/' + FILE_NAME)
                else:
                    shutil.copy(img, 'VAL/' + CLASS.upper() + '/' + FILE_NAME)

print("\nImágenes distribuidas correctamente.")

FileNotFoundError: [Errno 2] No such file or directory: 'archive/'

## 4. Carga del dataset en memoria

La función `load_data()` lee las imágenes con OpenCV y las almacena como arrays NumPy. Retorna las imágenes (`X`), las etiquetas numéricas (`y`) y un diccionario con las clases.

In [ ]:
def load_data(dir_path, img_size=(100, 100)):
    """
    Carga las imágenes como np.arrays y les cambia el tamaño.
    Retorna X (imágenes), y (etiquetas) y labels (diccionario de clases).
    """
    X = []
    y = []
    i = 0
    labels = dict()
    for path in tqdm(sorted(os.listdir(dir_path))):
        if not path.startswith('.'):
            labels[i] = path
            for file in os.listdir(dir_path + path):
                if not file.startswith('.'):
                    img = cv2.imread(dir_path + path + '/' + file)
                    X.append(img)
                    y.append(i)
            i += 1
    X = np.array(X, dtype=object)
    y = np.array(y)
    print(f'{len(X)} imágenes cargadas desde: {dir_path}.')
    return X, y, labels


TRAIN_DIR = 'TRAIN/'
TEST_DIR  = 'TEST/'
VAL_DIR   = 'VAL/'
IMG_SIZE  = (224, 224)

X_train, y_train, labels = load_data(TRAIN_DIR, IMG_SIZE)
X_test,  y_test,  _      = load_data(TEST_DIR,  IMG_SIZE)
X_val,   y_val,   _      = load_data(VAL_DIR,   IMG_SIZE)

print(f"\nEtiquetas: {labels}")

## 5. Visualización de muestras del dataset

Se visualizan 30 imágenes del conjunto de entrenamiento para inspeccionar la calidad y variedad antes de preprocesar.

In [ ]:
def plot_samples(X, y, labels, n=30):
    """
    Muestra n imágenes del conjunto dado, agrupadas por clase.
    """
    for index in labels:
        imgs = X[np.argwhere(y == index)][:n]
        j = 10
        i = int(np.ceil(len(imgs) / j))
        fig, axs = plt.subplots(i, j, figsize=(20, 10))
        axs = axs.flatten()
        for idx, im in enumerate(imgs):
            axs[idx].imshow(cv2.cvtColor(im[0], cv2.COLOR_BGR2RGB))
            axs[idx].axis('off')
        for ax in axs[len(imgs):]:
            ax.axis('off')
        fig.suptitle(f'Tumor: {labels[index].upper()}', fontsize=20)
        plt.tight_layout()
        plt.show()

plot_samples(X_train, y_train, labels, n=30)

## 6. Preprocesamiento: Recorte de imágenes (Crop)

La función `crop_imgs()` elimina el fondo innecesario:  
1. Convierte a escala de grises y aplica un umbral binario  
2. Encuentra el contorno más grande  
3. Detecta los puntos extremos y recorta la región de interés  

Esto permite al modelo enfocarse en el tejido cerebral.

In [ ]:
def crop_imgs(set_name, add_pixels_value=0):
    """
    Encuentra los puntos extremos de la imagen y la corta de forma rectangular.
    """
    set_new = []
    for img in set_name:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
        thresh = cv2.erode(thresh, None, iterations=2)
        thresh = cv2.dilate(thresh, None, iterations=2)

        cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = imutils.grab_contours(cnts)
        c = max(cnts, key=cv2.contourArea)

        extLeft  = tuple(c[c[:, :, 0].argmin()][0])
        extRight = tuple(c[c[:, :, 0].argmax()][0])
        extTop   = tuple(c[c[:, :, 1].argmin()][0])
        extBot   = tuple(c[c[:, :, 1].argmax()][0])

        ADD_PIXELS = add_pixels_value
        new_img = img[
            extTop[1] - ADD_PIXELS : extBot[1] + ADD_PIXELS,
            extLeft[0] - ADD_PIXELS : extRight[0] + ADD_PIXELS
        ].copy()
        set_new.append(new_img)
    return np.array(set_new, dtype=object)


X_train_crop = crop_imgs(set_name=X_train)
X_val_crop   = crop_imgs(set_name=X_val)
X_test_crop  = crop_imgs(set_name=X_test)

plot_samples(X_train_crop, y_train, labels, n=30)
print("Recorte aplicado correctamente.")

## 7. Guardar imágenes recortadas en disco

Las imágenes procesadas se guardan en las carpetas `TRAIN_CROP`, `VAL_CROP` y `TEST_CROP` para ser usadas por los generadores de datos de Keras.

In [ ]:
def save_new_images(x_set, y_set, folder_name):
    i = 0
    for (img, imclass) in zip(x_set, y_set):
        if imclass == 0:
            cv2.imwrite(folder_name + 'NO/' + str(i) + '.jpg', img)
        else:
            cv2.imwrite(folder_name + 'YES/' + str(i) + '.jpg', img)
        i += 1


# Crear carpetas destino
for split in ['TRAIN_CROP', 'TEST_CROP', 'VAL_CROP']:
    for cls in ['YES', 'NO']:
        os.makedirs(f'{split}/{cls}', exist_ok=True)

save_new_images(X_train_crop, y_train, folder_name='TRAIN_CROP/')
save_new_images(X_val_crop,   y_val,   folder_name='VAL_CROP/')
save_new_images(X_test_crop,  y_test,  folder_name='TEST_CROP/')

print("Imágenes recortadas guardadas en disco.")

## 8. Redimensionado y preprocesamiento VGG16 (224×224)

Se redimensionan todas las imágenes a **224×224** píxeles (requisito de VGG16) y se aplica la función `preprocess_input` de Keras, que normaliza los píxeles según las estadísticas de ImageNet.

In [ ]:
def preprocess_imgs(set_name, img_size):
    """
    Redimensiona y aplica preprocesamiento VGG16 a cada imagen.
    """
    set_new = []
    for img in set_name:
        img = cv2.resize(
            img,
            dsize=img_size,
            interpolation=cv2.INTER_CUBIC
        )
        set_new.append(preprocess_input(img))
    return np.array(set_new)


X_train_prep = preprocess_imgs(set_name=X_train_crop, img_size=IMG_SIZE)
X_test_prep  = preprocess_imgs(set_name=X_test_crop,  img_size=IMG_SIZE)
X_val_prep   = preprocess_imgs(set_name=X_val_crop,   img_size=IMG_SIZE)

print(f"Forma de X_train_prep: {X_train_prep.shape}")
print(f"Forma de X_val_prep:   {X_val_prep.shape}")
print(f"Forma de X_test_prep:  {X_test_prep.shape}")

plot_samples(X_train_prep, y_train, labels, n=30)

## 9. Data Augmentation con ImageDataGenerator

Dado el pequeño tamaño del dataset (~253 imágenes), se aplica **data augmentation** para mejorar la generalización del modelo.  
Keras permite hacerlo en tiempo real durante el entrenamiento usando `ImageDataGenerator` con generadores de flujo desde disco.

In [ ]:
TRAIN_DIR = 'TRAIN_CROP/'
VAL_DIR   = 'VAL_CROP/'

train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    brightness_range=[0.5, 1.5],
    horizontal_flip=True,
    vertical_flip=True,
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    color_mode='rgb',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='binary',
    seed=RANDOM_SEED
)

validation_generator = test_datagen.flow_from_directory(
    VAL_DIR,
    color_mode='rgb',
    target_size=IMG_SIZE,
    batch_size=16,
    class_mode='binary',
    seed=RANDOM_SEED
)

print("Generadores de datos configurados.")

## 10. Construcción del modelo: VGG16 + Transfer Learning

Se carga VGG16 preentrenado en ImageNet **sin la capa de clasificación** (`include_top=False`).  
Se congela la capa base para conservar los pesos aprendidos y se añade una cabeza de clasificación personalizada:  
- `Flatten` → `Dropout(0.5)` → `Dense(1, sigmoid)` (clasificación binaria)

In [ ]:
# Cargar modelo base VGG16 preentrenado (sin la parte de clasificación)
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=IMG_SIZE + (3,)
)

NUM_CLASSES = 1

model = Sequential()
model.add(base_model)
model.add(layers.Flatten())
model.add(layers.Dropout(0.5))
model.add(layers.Dense(NUM_CLASSES, activation='sigmoid'))

# Congelar los pesos del modelo base (Transfer Learning)
model.layers[0].trainable = False

model.compile(
    loss='binary_crossentropy',
    optimizer=RMSprop(learning_rate=1e-4),
    metrics=['accuracy']
)

model.summary()

## 11. Entrenamiento del modelo

Se entrena por un máximo de **30 épocas** usando los generadores con data augmentation.  
`EarlyStopping` detiene el entrenamiento si la accuracy no mejora en 6 épocas consecutivas, evitando el sobreajuste.

In [ ]:
EPOCHS = 30
batch_size = 32
val_batch_size = 16

es = EarlyStopping(
    monitor='accuracy',
    mode='max',
    patience=6
)

history = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // batch_size,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=len(X_test) // val_batch_size,
    callbacks=[es]
)

print("Entrenamiento completado.")

## 12. Visualización de la evolución del entrenamiento

Se grafican la **accuracy** y la **pérdida** a lo largo de las épocas para los conjuntos de entrenamiento y validación.

In [ ]:
acc      = history.history['accuracy']
val_acc  = history.history['val_accuracy']
loss     = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(history.epoch) + 1)

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc,     label='Train Set')
plt.plot(epochs_range, val_acc, label='Val Set')
plt.legend(loc='best')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.title('Evolución de la Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss,     label='Train Set')
plt.plot(epochs_range, val_loss, label='Val Set')
plt.legend(loc='best')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.title('Evolución de la Pérdida')

plt.tight_layout()
plt.show()
print(history.history.keys())

## 13. Evaluación del modelo: Matriz de Confusión

Se evalúa el modelo tanto en el conjunto de **validación** como en el de **prueba**.  
La función `plot_confusion_matrix()` muestra la distribución de predicciones correctas e incorrectas por clase.

In [ ]:
def plot_confusion_matrix(cm, classes, normalize=False,
                          title='Confusion matrix', cmap=plt.cm.Blues):
    """
    Imprime y grafica la matriz de confusión.
    Se puede aplicar normalización con normalize=True.
    """
    plt.figure(figsize=(6, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=90)
    plt.yticks(tick_marks, classes)
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    thresh = cm.max() / 2.
    cm = np.round(cm, 2)
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment='center',
                 color='white' if cm[i, j] > thresh else 'black')
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()


# --- Validación ---
predictions = model.predict(X_val_prep)
predictions = [1 if x > 0.5 else 0 for x in predictions]

accuracy = accuracy_score(y_val, predictions)
print(f'Val Accuracy = {accuracy:.2f}')

confusion_mtx = confusion_matrix(y_val, predictions)
cm = plot_confusion_matrix(confusion_mtx,
                           classes=list(labels.items()),
                           normalize=False)

# --- Prueba ---
predictions_test = model.predict(X_test_prep)
predictions_test = [1 if x > 0.5 else 0 for x in predictions_test]

accuracy_test = accuracy_score(y_test, predictions_test)
print(f'Test Accuracy = {accuracy_test:.2f}')

confusion_mtx_test = confusion_matrix(y_test, predictions_test)
cm_test = plot_confusion_matrix(confusion_mtx_test,
                                classes=list(labels.items()),
                                normalize=False)

## 14. Conclusiones

| Métrica | Valor obtenido |
|---|---|
| Val Accuracy  | ~78% |
| Test Accuracy | ~70% |

**Análisis:**  
- El modelo logra clasificar correctamente la mayoría de los casos, pero aún no alcanza el umbral de 80% deseado por el cliente.
- La diferencia entre validación y prueba sugiere cierto grado de sobreajuste.

**Posibles mejoras:**  
- Descongelar capas del modelo base para un **fine-tuning** más profundo  
- Probar otras arquitecturas: `ResNet50`, `EfficientNet`, `InceptionV3`  
- Aumentar el dataset o aplicar técnicas de augmentation más agresivas  
- Optimizar hiperparámetros: learning rate, tamaño de batch, dropout